# LifeMate — một notebook chạy trọn pipeline trên Kaggle

**Cách chạy:** Add Input bộ dữ liệu đã giải nén, bật **Internet**, chọn **GPU T4/P100** rồi Run All.
Không cần clone GitHub. Mặc định notebook fine-tune LLM trước trên TRAIN rồi dùng adapter vừa train. Notebook chứa toàn bộ mã nguồn: audio tiếng Việt → faster-whisper → Qwen → validate/call trên `ToolEnv` thật → submission + logs.

Bộ dữ liệu cần có `<split>.jsonl`, `<split>_tools_catalog.json`, audio và **zaic_tools đúng phiên bản của split**. Notebook chỉ đọc manifest/catalog/audio; trạng thái ứng dụng do `ToolEnv` quản lý.

Chỉnh cấu hình bên dưới. `LIMIT=0` chạy **toàn bộ split**. Model tải từ Hugging Face trước khi suy luận; có thể dùng model gắn sẵn ở `/kaggle/input`. Model ASR mặc định là `Systran/faster-whisper-small`; muốn dùng PhoWhisper, đặt `ASR_REPO="vinai/PhoWhisper-small"` và `ASR_FORMAT="transformers"` để tự chuyển sang CTranslate2.

Kết quả lưu trong `/kaggle/working/lifemate_runs/<RUN_NAME>/`: `submission.jsonl`, `diagnostics.jsonl`, `inference.log`, `setup.log`, `models.log`, `validation.json`, `run_config.json`, `environment.txt`, và `lifemate_artifacts.zip`. Logs chẩn đoán có transcript; kiểm tra trước khi chia sẻ. Notebook không bảo đảm giới hạn 30 giây/mẫu hoặc điểm thi chính thức.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, datetime, shutil, zipfile, hashlib

# ===== CẤU HÌNH — thông thường chỉ cần chỉnh DATA_ROOT =====
DATA_ROOT = None  # None: tự tìm trong /kaggle/input; hoặc "/kaggle/input/<dataset>/train+public_test"
SPLIT = "public_test"  # train / public_test / private_test
LIMIT = 0             # 0 = toàn bộ split; đặt 5 để smoke test
REQUIRE_GPU = True    # False cho CPU (rất chậm; không bảo đảm giới hạn thời gian)
GPU_INDEX = 0         # dùng một GPU; không cộng gộp VRAM của 2 T4

# Fine-tune trước rồi tự nạp adapter cho inference.
TRAIN_LLM = True
TRAIN_BACKEND = "unsloth"  # backend mặc định; cần CUDA
TRAIN_USE_ASR = False  # bật nếu train/history thiếu transcript
TRAIN_DATA_ROOT = None  # None: tự tìm train/train.jsonl trong /kaggle/input
LOCAL_ADAPTER = None   # TRAIN_LLM=False + đường dẫn adapter để dùng bản đã train
TRAIN_EPOCHS = 3
TRAIN_MAX_STEPS = -1   # -1: theo epochs; đặt 2 cho kiểm tra ngắn, không phải model đủ chất lượng
TRAIN_MAX_LENGTH = 8192
TRAIN_GRAD_ACCUMULATION = 8
TRAIN_VALIDATION_FRACTION = 0.1

ASR_REPO = "Systran/faster-whisper-small"
ASR_FORMAT = "ctranslate2"  # hoặc "transformers" (ví dụ vinai/PhoWhisper-small)
LLM_REPO = "Qwen/Qwen3-4B-Instruct-2507"
ASR_REVISION = None  # None: resolve main thành SHA và ghi lại trong model_manifest.json
LLM_REVISION = None
LOCAL_ASR = None     # thư mục CTranslate2 có model.bin; bỏ qua tải/chuyển ASR khi đã có
LOCAL_LLM = None     # thư mục Transformers có config.json + tokenizer + trọng số
HF_SECRET_NAME = None  # tùy chọn: tên Kaggle Secret chứa HF token (public models không cần)

SAMPLE_SECONDS = 28.0  # soft budget giống baseline, không phải hard timeout
MAX_INPUT_TOKENS = 6000
MAX_NEW_TOKENS = 192
MAX_CALLS = 8
MAX_INVALID = 3
MAX_TOOL_DETAILS = 35
ASR_BEAM = 1
SEED = 42
FAIL_FAST = False      # False: lưu lỗi từng mẫu và tiếp tục, báo lỗi ở bước kiểm chứng
USE_TRAIN_TRANSCRIPT = False  # chỉ được bật cho split train
RUN_TRAIN_SCORE = True        # proxy F1, chỉ chạy khi SPLIT == "train"
RUN_NAME = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")

APP = Path("/kaggle/working/lifemate_app")
MODEL_ROOT = Path("/kaggle/working/lifemate_models")
OUT = Path("/kaggle/working/lifemate_runs") / RUN_NAME
for folder in (APP / "lifemate", APP / "tests", MODEL_ROOT, OUT):
    folder.mkdir(parents=True, exist_ok=True)

def run_logged(command, log_path, *, env=None, cwd=None):
    # Không log env/token hoặc dùng shell=True. Giữ đúng exit status của subprocess.
    with Path(log_path).open("a", encoding="utf-8", buffering=1) as log:
        with subprocess.Popen([str(x) for x in command], cwd=cwd, env=env,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, errors="replace", bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            status = process.wait()
    if status:
        raise RuntimeError(f"Command failed (exit {status}); xem {log_path}")

assert SPLIT in {"train", "public_test", "private_test"}
assert isinstance(LIMIT, int) and LIMIT >= 0
assert not USE_TRAIN_TRANSCRIPT or SPLIT == "train"
assert ASR_FORMAT in {"ctranslate2", "transformers"}
print("Artifacts:", OUT)


## 1. Dependency và runtime

Giữ nguyên PyTorch/CUDA do Kaggle cung cấp, không cài đè torch. Các thư viện ASR/LLM được cài trong tiến trình riêng trước khi dùng. CTranslate2 cần CUDA 12 + cuDNN 9 khi chạy GPU; bước suy luận thực tế sẽ kiểm tra việc nạp model. Nếu GPU memory không đủ, giảm `MAX_INPUT_TOKENS`, chọn model nhỏ hơn hoặc đổi GPU.


In [ ]:
packages = [
    "numpy>=1.26,<3", "transformers==4.57.3", "accelerate==1.15.0",
    "safetensors==0.8.0", "faster-whisper==1.2.1", "av==16.1.0", "ctranslate2==4.6.0",
    "huggingface-hub==0.36.2", "pytest>=8,<10", "peft==0.19.1", "bitsandbytes>=0.45.5,<0.50,!=0.46.0,!=0.48.0",
    "unsloth==2025.12.10", "unsloth-zoo==2025.12.8",
]
# Kiểm tra torch trước: tránh pip tự kéo một bản CUDA khác với Kaggle.
run_logged([sys.executable, "-c", "import torch; print('Kaggle torch:', torch.__version__)"], OUT / "setup.log")
# Constrain the preinstalled CUDA stack so dependency resolution cannot replace it.
import importlib.metadata
stack = []
for distribution in ("torch", "torchvision", "triton"):
    try:
        stack.append(f"{distribution}=={importlib.metadata.version(distribution)}")
    except importlib.metadata.PackageNotFoundError:
        pass
constraint = OUT / "kaggle_cuda_constraints.txt"
constraint.write_text("\n".join(stack) + "\n", encoding="utf-8")
run_logged([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--constraint", constraint,
            "--upgrade-strategy", "only-if-needed", *packages], OUT / "setup.log")
BASE_ENV = os.environ.copy()
BASE_ENV.update(PYTHONUNBUFFERED="1", PYTHONDONTWRITEBYTECODE="1",
                TOKENIZERS_PARALLELISM="false", CUBLAS_WORKSPACE_CONFIG=":4096:8")
BASE_ENV["CUDA_VISIBLE_DEVICES"] = str(GPU_INDEX)
# Expose pip-provided NVIDIA libraries to the child process's dynamic loader.
import site
lib_dirs = []
for site_dir in site.getsitepackages():
    lib_dirs.extend(str(p) for p in Path(site_dir).glob("nvidia/*/lib") if p.is_dir())
BASE_ENV["LD_LIBRARY_PATH"] = os.pathsep.join(lib_dirs + [BASE_ENV.get("LD_LIBRARY_PATH", "")])
hardware_script = """
import json, torch, ctranslate2
from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
from faster_whisper import WhisperModel
gpu = torch.cuda.is_available()
print(json.dumps(dict(torch=torch.__version__, cuda=torch.version.cuda, gpu=gpu,
    gpu_name=torch.cuda.get_device_name(0) if gpu else None,
    bf16=torch.cuda.is_bf16_supported() if gpu else False,
    ct2=ctranslate2.__version__)))
"""
hardware = subprocess.run([sys.executable, "-c", hardware_script], env=BASE_ENV,
                          capture_output=True, text=True)
(OUT / "hardware.log").write_text(hardware.stdout + hardware.stderr, encoding="utf-8")
if hardware.returncode:
    raise RuntimeError("Import dependency thất bại; xem hardware.log:\n" + hardware.stderr[-4000:])
HW = json.loads(hardware.stdout.strip().splitlines()[-1])
print(HW)
if REQUIRE_GPU and not HW["gpu"]:
    raise RuntimeError("Bật GPU trong Kaggle Settings rồi Run All, hoặc đặt REQUIRE_GPU=False.")
DEVICE = "cuda" if HW["gpu"] else "cpu"
LLM_DTYPE = ("bfloat16" if HW["bf16"] else "float16") if HW["gpu"] else "float32"
ASR_COMPUTE = "float16" if HW["gpu"] else "int8"
run_logged([sys.executable, "-m", "pip", "freeze"], OUT / "environment.txt")


## 2. Tìm và kiểm tra dữ liệu

Nếu có nhiều bản của cùng split, đặt `DATA_ROOT` rõ ràng. `DATA_ROOT` có thể là thư mục cha hoặc chính thư mục split. Không cần chép dữ liệu từ vùng input chỉ đọc. Chỉ đọc manifest và kiểm tra đường dẫn audio; không duyệt nội dung các thư mục `*_environment`.


In [ ]:
def discover_split(root, split):
    candidates = []
    for current, dirs, files in os.walk(root):
        dirs[:] = sorted(d for d in dirs if not (
            d.endswith("_environment") or d in {"audio", "__MACOSX", ".git"}
        ))
        if f"{split}.jsonl" in files:
            candidates.append(Path(current))
            dirs[:] = []
    return sorted(set(p.resolve() for p in candidates))

if DATA_ROOT:
    location = Path(DATA_ROOT).expanduser().resolve()
    if (location / SPLIT / f"{SPLIT}.jsonl").is_file():
        SPLIT_DIR = location / SPLIT
    elif (location / f"{SPLIT}.jsonl").is_file():
        SPLIT_DIR = location
    else:
        raise FileNotFoundError(f"Không tìm thấy {SPLIT}.jsonl trong {location}")
else:
    candidates = discover_split("/kaggle/input", SPLIT)
    if len(candidates) != 1:
        raise ValueError(f"Tìm thấy {len(candidates)} split: {candidates}. Đặt DATA_ROOT rõ ràng.")
    SPLIT_DIR = candidates[0]
CATALOG_PATH = SPLIT_DIR / f"{SPLIT}_tools_catalog.json"
if not CATALOG_PATH.is_file():
    raise FileNotFoundError(CATALOG_PATH)
if not ((SPLIT_DIR / "zaic_tools.py").is_file() or (SPLIT_DIR / "zaic_tools").is_dir()):
    raise FileNotFoundError(f"Cần zaic_tools đi kèm split tại {SPLIT_DIR}, không dùng package khác trên pip.")
ITEMS = []
with (SPLIT_DIR / f"{SPLIT}.jsonl").open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            ITEMS.append(json.loads(line))
if LIMIT:
    ITEMS = ITEMS[:LIMIT]
EXPECTED_IDS = [item["id"] for item in ITEMS]
if not ITEMS or len(set(EXPECTED_IDS)) != len(EXPECTED_IDS):
    raise ValueError("Manifest rỗng hoặc có id trùng trong phần cần chạy.")
for item in ITEMS:
    audio = (SPLIT_DIR / item["audio"]).resolve()
    if not audio.is_relative_to(SPLIT_DIR.resolve()) or not audio.is_file():
        raise FileNotFoundError(f"Audio không hợp lệ cho {item['id']}: {audio}")
print(f"Split: {SPLIT_DIR}; samples cần chạy: {len(ITEMS)}")


## 3. Toàn bộ mã pipeline nhúng trong notebook

Các cell dưới ghi mã vào `/kaggle/working/lifemate_app` để chạy trong subprocess, tách runtime tải model (online) khỏi suy luận (local-only). Logic baseline được giữ nguyên; không cần repository bên ngoài. Có cả mã fine-tune, bộ unit test và scorer train.


### `lifemate/__init__.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/lifemate/__init__.py
"""Offline LifeMate tool-using agent baseline."""


### `lifemate/catalog.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/lifemate/catalog.py
"""Catalog parsing. Never accesses any *_environment/ file."""
from __future__ import annotations

import json
import re
import unicodedata
from pathlib import Path
from typing import Any


def load_catalog(path: Path) -> dict[str, dict]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    tools: dict[str, dict] = {}
    for app in payload.get("apps", []):
        for spec in app.get("tools", []):
            name = spec.get("name")
            if isinstance(name, str) and name:
                tools[name] = spec
    # The competition makes this meta tool universally available. Its schema
    # is supplied here only if missing from the catalog; env.validate decides.
    tools.setdefault("sys.ask_missing_info", {
        "name": "sys.ask_missing_info",
        "kind": "meta",
        "description": "Yêu cầu thông tin còn thiếu cho công cụ dự định gọi",
        "parameters": {"type": "object", "properties": {
            "tool_name": {"type": "string"},
            "params": {"type": "array", "items": {"type": "string"}},
        }, "required": ["tool_name", "params"]},
    })
    return tools


def allowed_tools(catalog: dict[str, dict], installed_apps: Any) -> dict[str, dict]:
    """Only the tools exposed to the sample, plus sys.ask_missing_info."""
    explicit: set[str] = set()
    allowed_apps: set[str] = set()
    if isinstance(installed_apps, dict):
        for app, info in installed_apps.items():
            allowed_apps.add(str(app))
            if isinstance(info, dict) and "tools" in info:
                for entry in info["tools"] or []:
                    name = entry if isinstance(entry, str) else entry.get("name") if isinstance(entry, dict) else None
                    if name:
                        explicit.add(str(name))
            else:
                explicit.update(name for name in catalog if name.startswith(f"{app}."))
    elif isinstance(installed_apps, list):
        for item in installed_apps:
            if isinstance(item, str):
                allowed_apps.add(item)
                explicit.update(name for name in catalog if name.startswith(f"{item}."))
            elif isinstance(item, dict):
                app = item.get("id", item.get("name"))
                if not app:
                    continue
                allowed_apps.add(str(app))
                if "tools" in item:
                    for entry in item["tools"] or []:
                        name = entry if isinstance(entry, str) else entry.get("name") if isinstance(entry, dict) else None
                        if name:
                            explicit.add(str(name))
                else:
                    explicit.update(name for name in catalog if name.startswith(f"{app}."))
    result = {name: catalog[name] for name in sorted(explicit) if name in catalog and name.split(".")[0] in allowed_apps}
    result["sys.ask_missing_info"] = catalog["sys.ask_missing_info"]
    return result


def _tokens(s: str) -> set[str]:
    norm = unicodedata.normalize("NFD", s.casefold())
    norm = "".join(c for c in norm if not unicodedata.combining(c))
    return {t for t in re.findall(r"[a-z0-9_]+", norm) if len(t) > 1}


def rank_tools(allowed: dict[str, dict], query: str, last_tool: str | None = None) -> list[str]:
    q = _tokens(query)
    last_app = last_tool.split(".")[0] if last_tool else None
    ranks = []
    for name, spec in allowed.items():
        n = _tokens(name.replace("_", " ").replace(".", " "))
        d = _tokens(spec.get("description", ""))
        props = _tokens(" ".join(spec.get("parameters", {}).get("properties", {}).keys()))
        score = 4 * len(q & n) + 1.6 * len(q & d) + 0.7 * len(q & props)
        if last_app and name.startswith(f"{last_app}."):
            score += 2
        if spec.get("kind") == "read":
            score += 0.3
        if name == "sys.ask_missing_info":
            score += 0.4
        ranks.append((-score, name))
    return [name for _, name in sorted(ranks)]


def compact_schema(spec: dict) -> dict:
    params = spec.get("parameters", {})
    return {
        "name": spec["name"],
        "kind": spec.get("kind", ""),
        "description": str(spec.get("description", ""))[:250],
        "parameters": params,
        "returns": str(spec.get("returns", ""))[:220],
    }


def prompt_catalog(allowed: dict[str, dict], query: str, last_tool: str | None, max_detailed: int) -> dict:
    ordered = rank_tools(allowed, query, last_tool)
    detailed = set(ordered[:max_detailed])
    detailed.add("sys.ask_missing_info")
    # Read tools from selected apps help discover IDs before attempting writes.
    for name in ordered[:max(1, max_detailed // 2)]:
        app = name.split(".")[0]
        reads = [x for x in ordered if x.startswith(f"{app}.") and allowed[x].get("kind") == "read"]
        for read in reads[:2]:
            if len(detailed) < max_detailed + 8:
                detailed.add(read)
    return {
        "tool_index": [{"name": n, "kind": allowed[n].get("kind", ""),
                        "description": str(allowed[n].get("description", ""))[:105]} for n in sorted(allowed)],
        "tool_schemas": [compact_schema(allowed[n]) for n in ordered if n in detailed],
    }


### `lifemate/models.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/lifemate/models.py
"""Two completely local, deterministic model wrappers."""
from __future__ import annotations

from collections import OrderedDict
from pathlib import Path


class LocalASR:
    def __init__(self, model_path: Path, device: str = "cuda", compute_type: str = "float16", beam_size: int = 1):
        from faster_whisper import WhisperModel
        if not model_path.is_dir():
            raise FileNotFoundError(f"ASR model directory not found: {model_path}")
        self.model = WhisperModel(
            str(model_path), device=device, compute_type=compute_type,
            local_files_only=True, cpu_threads=4,
        )
        self.beam_size = beam_size
        self.cache: OrderedDict[str, str] = OrderedDict()

    def transcribe(self, path: Path) -> str:
        key = str(path.resolve())
        if key in self.cache:
            self.cache.move_to_end(key)
            return self.cache[key]
        if not path.is_file():
            raise FileNotFoundError(f"Audio file not found: {path}")
        segments, _ = self.model.transcribe(
            str(path), language="vi", task="transcribe", beam_size=self.beam_size,
            temperature=0, condition_on_previous_text=False, vad_filter=False,
        )
        result = " ".join(seg.text.strip() for seg in segments).strip()
        self.cache[key] = result
        if len(self.cache) > 512:
            self.cache.popitem(last=False)
        return result


class LocalLLM:
    def __init__(self, model_path: Path, device: str = "cuda", dtype: str = "float16",
                 max_input_tokens: int = 6000, max_new_tokens: int = 192,
                 adapter_path: Path | None = None):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        if not model_path.is_dir():
            raise FileNotFoundError(f"LLM model directory not found: {model_path}")
        self.torch = torch
        tokenizer_path = adapter_path if adapter_path and (adapter_path / "tokenizer_config.json").is_file() else model_path
        self.tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path), local_files_only=True, trust_remote_code=False)
        self.tokenizer.truncation_side = "right"
        torch_dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[dtype]
        self.model = AutoModelForCausalLM.from_pretrained(
            str(model_path), torch_dtype=torch_dtype,
            local_files_only=True, trust_remote_code=False,
            low_cpu_mem_usage=True,
            attn_implementation="sdpa",
        )
        if adapter_path is not None:
            if not (adapter_path / "adapter_config.json").is_file():
                raise FileNotFoundError(f"LoRA adapter not found: {adapter_path}")
            from peft import PeftModel
            self.model = PeftModel.from_pretrained(self.model, str(adapter_path),
                                                 local_files_only=True, is_trainable=False)
        self.model.to(device)
        self.model.eval()
        self.device = device
        self.max_input_tokens = min(max_input_tokens, 16000)
        self.max_new_tokens = min(max_new_tokens, 384)

    def generate(self, system: str, user: str) -> str:
        tokenizer = self.tokenizer
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        encoded = tokenizer(rendered, return_tensors="pt", truncation=True,
                            max_length=self.max_input_tokens, return_token_type_ids=False)
        encoded = {k: v.to(self.device) for k, v in encoded.items()}
        with self.torch.inference_mode():
            ids = self.model.generate(
                **encoded, max_new_tokens=self.max_new_tokens,
                do_sample=False, num_beams=1,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
        completion = ids[0, encoded["input_ids"].shape[1]:]
        return tokenizer.decode(completion, skip_special_tokens=True).strip()


### `lifemate/agent.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/lifemate/agent.py
"""Observe -> plan a single tool call -> validate -> execute -> observe."""
from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from .catalog import allowed_tools, prompt_catalog

SYSTEM_PROMPT = """Bạn là AI Agent LifeMate. Chỉ xuất ĐÚNG MỘT JSON object, KHÔNG markdown hoặc diễn giải.
Bạn có transcript yêu cầu nói tiếng Việt, context, history, các tool được cài và kết quả các calls trước.
Mục tiêu: hoàn tất đúng trạng thái ứng dụng với ít calls hợp lý nhất.

Một trong hai dạng output:
{"action":"call","name":"app.tool","arguments":{"key":"value"}}
{"action":"done"}

QUY TẮC BẮT BUỘC:
- Mỗi lần chỉ đề xuất 1 tool call; bạn sẽ được xem kết quả và quyết định tiếp.
- Chỉ gọi tools trong tool_schemas, đúng tên và JSON types/enum/required/x-one-of.
- Với x-ref, trước tiên dùng read tool tương ứng để tìm ID thực; KHÔNG bịa ID.
- Nếu cần kết hợp nhiều hành động, hoàn tất từng hành động theo thứ tự phụ thuộc.
- Dùng user context (current_time/timezone/location/profile) và lịch sử để giải quyết tham chiếu.
- Đừng thay đổi trạng thái ngoài mục tiêu của người dùng; đừng gọi read tools lặp vô ích.
- Nếu thiếu thông tin bắt buộc không suy ra được qua context/read tools, gọi
  sys.ask_missing_info với {"tool_name":"tool dự kiến", "params":["tham số còn thiếu"]}, rồi dừng.
- Nếu không ứng dụng được cài nào có khả năng đáp ứng yêu cầu: xuất {"action":"done"} ngay, KHÔNG gọi ask_missing_info.
- Nếu đã hoàn tất: {"action":"done"}. Không tự nói là đã gọi tool.
- Arguments must be a JSON object. Output exactly one JSON object.
"""


def parse_decision(raw: str) -> dict:
    """Find first JSON object, including model outputs with harmless surrounding text."""
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL)
    decoder = json.JSONDecoder()
    for idx, char in enumerate(raw):
        if char != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(raw[idx:])
        except json.JSONDecodeError:
            continue
        if not isinstance(obj, dict):
            continue
        if "tool_calls" in obj and isinstance(obj["tool_calls"], list) and obj["tool_calls"]:
            obj = obj["tool_calls"][0]
        if obj.get("action") in ("done", "finish", "stop", "none"):
            return {"action": "done"}
        if obj.get("action") == "call" or ("name" in obj and "arguments" in obj):
            return {"action": "call", "name": obj.get("name"), "arguments": obj.get("arguments")}
    raise ValueError(f"Model did not produce a valid tool decision: {raw[:300]}")


def short_json(obj: Any, limit: int = 5000) -> str:
    s = json.dumps(obj, ensure_ascii=False, default=str, separators=(",", ":"))
    return s if len(s) <= limit else s[:limit] + "...(truncated)"


def fit_prompt_payload(info: dict, max_chars: int = 20000) -> str:
    """Fit large catalogs while preserving syntactically valid JSON at all times."""
    # The ranker puts likely-relevant schemas first. Always retain meta schema.
    from copy import deepcopy
    payload = deepcopy(info)
    cat = payload["available_tools"]

    def dumped():
        return json.dumps(payload, ensure_ascii=False, default=str, separators=(",", ":"))

    s = dumped()
    while len(s) > max_chars and len(cat["tool_schemas"]) > 2:
        # Remove lowest-ranked non-meta schema.
        i = next((i for i in range(len(cat["tool_schemas"]) - 1, -1, -1)
                  if cat["tool_schemas"][i]["name"] != "sys.ask_missing_info"), None)
        if i is None:
            break
        cat["tool_schemas"].pop(i)
        s = dumped()
    if len(s) > max_chars:
        for desc in cat["tool_index"]:
            desc["description"] = desc.get("description", "")[:35]
        s = dumped()
    if len(s) > max_chars:
        # Preserve tool names but make previous observations concise.
        for o in payload["previous_actions_and_results"]:
            if isinstance(o.get("result"), str):
                o["result"] = o["result"][:1200]
        s = dumped()
    if len(s) > max_chars:
        payload["conversation_history"] = payload["conversation_history"][-3:]
        s = dumped()
    if len(s) > max_chars:
        # Extremely large catalogs: collapse descriptions in index first.
        cat["tool_index"] = [{"name": x["name"]} for x in cat["tool_index"]]
        s = dumped()
    # Do not truncate into an invalid JSON document; if the request itself
    # is enormous, the tokenizer max_input_tokens remains a last-resort limit.
    return s


def _train_transcript(item: dict) -> str | None:
    """Opt-in, TRAIN ONLY. Never read labels or transcripts from public/private test."""
    txt = item.get("audio_text")
    if isinstance(txt, str):
        return txt
    if isinstance(txt, dict):
        for key in ("latest", "current", "text", "transcript", "audio_text"):
            if isinstance(txt.get(key), str):
                return txt[key]
    if isinstance(txt, list):
        for entry in reversed(txt):
            if isinstance(entry, str):
                return entry
            if isinstance(entry, dict):
                for k in ("audio_text", "text", "transcript", "content"):
                    if isinstance(entry.get(k), str):
                        return entry[k]
    return None


def audio_path(split_dir: Path, relative: str) -> Path:
    path = (split_dir / relative).resolve()
    if not path.is_relative_to(split_dir.resolve()):
        raise ValueError(f"Audio path escapes split directory: {relative}")
    return path


def history_text(history: Any, split_dir: Path, asr: Any, start: float, time_limit: float) -> list[dict]:
    result = []
    if not isinstance(history, list):
        return result
    for turn in history[-12:]:
        if not isinstance(turn, dict):
            continue
        role = str(turn.get("role", turn.get("speaker", "user")))
        text = next((turn[k] for k in ("audio_text", "text", "content", "transcript")
                     if isinstance(turn.get(k), str) and turn[k].strip()), "")
        if not text and role.lower() in ("user", "human"):
            path = turn.get("audio", turn.get("audio_path"))
            if isinstance(path, str) and time.monotonic() - start < time_limit - 5:
                text = asr.transcribe(audio_path(split_dir, path))
        if text:
            result.append({"role": role, "text": text[:1700]})
    return result


@dataclass
class AgentConfig:
    max_calls: int = 8
    max_invalid: int = 3
    max_tool_details: int = 35
    sample_seconds: float = 28.0
    # Reserving time is best effort, not a hard CUDA generation timeout.
    reserve_seconds: float = 1.0
    use_train_transcript: bool = False


def build_step_prompt(transcript, context, history, enabled, observations, last_tool, max_tool_details):
    """Identical candidate selection and prompt payload for SFT and inference."""
    search_query = transcript + " " + " ".join(
        str(o.get("name", "")) + " " + short_json(o.get("result", ""), 900)
        for o in observations[-2:]
    )
    tools = prompt_catalog(enabled, search_query, last_tool, max_tool_details)
    info = {
        "request_transcript": transcript,
        "user_context": {k: v for k, v in context.items() if k != "installed_apps"},
        "conversation_history": history,
        "available_tools": tools,
        "previous_actions_and_results": observations[-7:],
    }
    return tools, fit_prompt_payload(info)


class LifeMateAgent:
    def __init__(self, asr: Any, llm: Any, catalog: dict[str, dict], cfg: AgentConfig):
        self.asr = asr
        self.llm = llm
        self.catalog = catalog
        self.cfg = cfg

    def run(self, item: dict, env: Any, split_dir: Path, split: str) -> dict:
        """env is created exactly once by caller; never instantiate another here."""
        t0 = time.monotonic()
        context = item.get("context", {})
        enabled = allowed_tools(self.catalog, context.get("installed_apps", {}))
        if len(enabled) == 1:  # only sys.ask_missing_info; no app can act
            return {"transcript": "", "elapsed_seconds": 0, "executed": 0, "invalid_proposals": 0}
        transcript = _train_transcript(item) if self.cfg.use_train_transcript and split == "train" else None
        if transcript is None:
            transcript = self.asr.transcribe(audio_path(split_dir, item["audio"]))
        history = history_text(item.get("history", []), split_dir, self.asr, t0, self.cfg.sample_seconds)
        observations: list[dict] = []
        invalid = 0
        executed = 0
        last_tool: str | None = None
        epoch = 0
        recent_reads: set[tuple[int, str, str]] = set()
        while executed < self.cfg.max_calls and invalid < self.cfg.max_invalid:
            if time.monotonic() - t0 > self.cfg.sample_seconds - self.cfg.reserve_seconds:
                break
            tools, user_prompt = build_step_prompt(
                transcript, context, history, enabled, observations, last_tool, self.cfg.max_tool_details)
            raw = self.llm.generate(SYSTEM_PROMPT, user_prompt)
            try:
                decision = parse_decision(raw)
            except ValueError as exc:
                invalid += 1
                observations.append({"error": str(exc)[:450], "instruction": "Trả về JSON hợp lệ."})
                continue
            if decision["action"] == "done":
                break
            name, args = decision.get("name"), decision.get("arguments")
            if not isinstance(name, str) or name not in enabled or name not in {x["name"] for x in tools["tool_schemas"]}:
                invalid += 1
                observations.append({"error": f"Tool {name!r} not exposed in tool_schemas; choose only from listed schemas."})
                continue
            if not isinstance(args, dict):
                invalid += 1
                observations.append({"error": "arguments must be a JSON object"})
                continue
            if name == "sys.ask_missing_info":
                if args.get("tool_name") not in enabled or args.get("tool_name") == name:
                    invalid += 1
                    observations.append({"error": "Missing-info tool_name must be another available tool."})
                    continue
            try:
                validation = env.validate(name, args)
            except Exception as exc:
                validation = {"ok": False, "error": repr(exc)}
            if not isinstance(validation, dict) or not validation.get("ok", False):
                invalid += 1
                observations.append({"error": "Schema invalid: " + short_json(validation, 1300), "name": name})
                continue
            read_key = (epoch, name, short_json(args))
            if enabled[name].get("kind") == "read" and read_key in recent_reads:
                invalid += 1
                observations.append({"error": "Duplicate read without intervening state change", "name": name})
                continue
            # This is the *only* place execution takes place. Calls that fail are logged by ToolEnv.
            try:
                result = env.call(name, args)
            except Exception as exc:
                result = {"ok": False, "error": repr(exc)}
            executed += 1
            last_tool = name
            if enabled[name].get("kind") == "read":
                recent_reads.add(read_key)
            elif enabled[name].get("kind") == "write":
                epoch += 1
                recent_reads.clear()
            observations.append({"name": name, "arguments": args, "result": short_json(result, 4500)})
            if name == "sys.ask_missing_info":
                break
        return {"transcript": transcript, "elapsed_seconds": round(time.monotonic() - t0, 3),
                "executed": executed, "invalid_proposals": invalid}


### `main.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/main.py
#!/usr/bin/env python3
"""LifeMate offline baseline: configurable data/model/output paths."""
from __future__ import annotations

import argparse
import json
import logging
import os
import random
import sys
import tempfile
import time
from pathlib import Path

# Must be set before loading any model. No network at inference.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

from lifemate.catalog import load_catalog
from lifemate.agent import AgentConfig, LifeMateAgent

LOG = logging.getLogger("lifemate")


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Offline LifeMate ASR -> LLM -> ToolEnv baseline")
    p.add_argument("--data-root", type=Path, required=True,
                   help="Folder holding <split>/ OR the split folder itself (e.g. /data in Docker)")
    p.add_argument("--split", choices=["train", "public_test", "private_test"], default="public_test")
    p.add_argument("--asr-model", type=Path, required=True, help="Local CTranslate2 Whisper directory")
    p.add_argument("--llm-model", type=Path, required=True, help="Local HF Transformers LLM directory")
    p.add_argument("--llm-adapter", type=Path, default=None, help="Optional local fine-tuned LoRA adapter")
    p.add_argument("--output", type=Path, default=Path("result/submission.jsonl"))
    p.add_argument("--log-path", type=Path, default=None, help="Optional diagnostic JSONL, never submitted")
    p.add_argument("--device", choices=["cuda", "cpu"], default="cuda")
    p.add_argument("--asr-compute-type", default="float16", choices=["float16", "int8_float16", "int8", "float32"])
    p.add_argument("--llm-dtype", default="bfloat16", choices=["float16", "bfloat16", "float32"])
    p.add_argument("--asr-beam", type=int, default=1)
    p.add_argument("--max-input-tokens", type=int, default=6000)
    p.add_argument("--max-new-tokens", type=int, default=192)
    p.add_argument("--max-calls", type=int, default=8)
    p.add_argument("--max-invalid", type=int, default=3)
    p.add_argument("--max-tool-details", type=int, default=35)
    p.add_argument("--sample-seconds", type=float, default=28.0)
    p.add_argument("--use-train-transcript", action="store_true", help="TRAIN ONLY: ablate ASR with train audio_text")
    p.add_argument("--limit", type=int, default=0, help="0 means run entire split; debugging only")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--fail-fast", action="store_true")
    return p.parse_args()


def seed_everything(seed: int) -> None:
    random.seed(seed)
    import numpy as np
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def resolve_data_layout(data_root: Path, split: str):
    """Return (split_dir, ToolEnv-root, temp owner). Symlink solves /data being split itself.

    Only creates a directory link; never opens/reads/copies/parses environment files.
    """
    location = data_root.expanduser().resolve()
    if (location / split / f"{split}.jsonl").is_file():
        return location / split, location, None
    if (location / f"{split}.jsonl").is_file():
        holder = tempfile.TemporaryDirectory(prefix="lifemate_split_root_")
        alias = Path(holder.name) / split
        alias.symlink_to(location, target_is_directory=True)
        return location, Path(holder.name), holder
    raise FileNotFoundError(
        f"Cannot locate {split}.jsonl under {location} or {location / split}. "
        "Point --data-root either to the split or to its parent."
    )


def main() -> None:
    args = parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    split_dir, env_root, _temp_holder = resolve_data_layout(args.data_root, args.split)
    # Import only the zaic_tools version shipped with this exact split.
    sys.path.insert(0, str(split_dir))
    from zaic_tools import ToolEnv
    seed_everything(args.seed)
    catalog = load_catalog(split_dir / f"{args.split}_tools_catalog.json")
    from lifemate.models import LocalASR, LocalLLM
    asr = LocalASR(args.asr_model, device=args.device, compute_type=args.asr_compute_type, beam_size=args.asr_beam)
    llm = LocalLLM(args.llm_model, device=args.device, dtype=args.llm_dtype,
                   max_input_tokens=args.max_input_tokens, max_new_tokens=args.max_new_tokens,
                   adapter_path=args.llm_adapter)
    cfg = AgentConfig(max_calls=args.max_calls, max_invalid=args.max_invalid,
                      max_tool_details=args.max_tool_details, sample_seconds=args.sample_seconds,
                      use_train_transcript=args.use_train_transcript)
    if args.use_train_transcript and args.split != "train":
        raise ValueError("--use-train-transcript is forbidden for test splits")
    agent = LifeMateAgent(asr=asr, llm=llm, catalog=catalog, cfg=cfg)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    if args.log_path:
        args.log_path.parent.mkdir(parents=True, exist_ok=True)
    input_file = split_dir / f"{args.split}.jsonl"
    overall_start = time.monotonic()
    count = 0
    # Write progressively, one line per sample. A sample's env is created exactly once.
    with input_file.open(encoding="utf-8") as fin, args.output.open("w", encoding="utf-8") as fout:
        log_file = args.log_path.open("w", encoding="utf-8") if args.log_path else None
        try:
            for line in fin:
                if not line.strip():
                    continue
                if args.limit and count >= args.limit:
                    break
                item = json.loads(line)
                sample_start = time.monotonic()
                env = ToolEnv(args.split, item["id"], root=str(env_root))
                try:
                    report = agent.run(item, env, split_dir, args.split)
                except Exception as exc:
                    LOG.exception("Sample %s encountered an error", item["id"])
                    report = {"error": repr(exc)}
                    if args.fail_fast:
                        raise
                calls = env.get_tool_calls()
                state_hash = env.get_state_hash()
                record = {"id": item["id"], "tool_calls": calls, "state_hash": state_hash}
                fout.write(json.dumps(record, ensure_ascii=False) + "\n")
                fout.flush()
                count += 1
                sample_time = time.monotonic() - sample_start
                LOG.info("%d %s: %d calls, %.2fs", count, item["id"], len(calls), sample_time)
                if log_file:
                    log_file.write(json.dumps({"id": item["id"], "sample_seconds": round(sample_time, 3), **report}, ensure_ascii=False) + "\n")
                    log_file.flush()
        finally:
            if log_file:
                log_file.close()
    LOG.info("Finished %d samples in %.1fs. Submission: %s", count, time.monotonic()-overall_start, args.output)


if __name__ == "__main__":
    main()


### `score_train.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/score_train.py
#!/usr/bin/env python3
"""Train-only proxy F1 for tool calls; NOT the official state-hash-gated score.

This script reads labeled TRAIN JSONL and submitted predictions, NOT app state.
It does not instantiate any ToolEnv or access *_environment/.
"""
from __future__ import annotations

import argparse
import json
import unicodedata
from collections import Counter
from pathlib import Path

from lifemate.catalog import load_catalog


def token_f1(a, b) -> float:
    aa = unicodedata.normalize("NFC", str(a).lower()).split()
    bb = unicodedata.normalize("NFC", str(b).lower()).split()
    if not aa and not bb:
        return 1.0
    if not aa or not bb:
        return 0.0
    common = sum((Counter(aa) & Counter(bb)).values())
    return 2 * common / (len(aa) + len(bb))


def calls_match(pred: dict, gold: dict, catalog: dict[str, dict]) -> bool:
    if pred.get("name") != gold.get("name"):
        return False
    p, g = pred.get("arguments", {}), gold.get("arguments", {})
    if not isinstance(p, dict) or not isinstance(g, dict) or set(p) != set(g):
        return False
    props = catalog.get(pred["name"], {}).get("parameters", {}).get("properties", {})
    for key in p:
        if props.get(key, {}).get("x-match") == "fuzzy":
            if token_f1(p[key], g[key]) <= 0.6:
                return False
        elif p[key] != g[key] or type(p[key]) is not type(g[key]):
            return False
    return True


def max_matches(preds: list, golds: list, catalog: dict) -> int:
    # Maximum one-to-one bipartite matching (order-independent).
    edges = [[j for j, gold in enumerate(golds) if calls_match(pred, gold, catalog)]
             for pred in preds]
    gold_to_pred: dict[int, int] = {}

    def augment(pred_idx: int, seen: set[int]) -> bool:
        for j in edges[pred_idx]:
            if j in seen:
                continue
            seen.add(j)
            if j not in gold_to_pred or augment(gold_to_pred[j], seen):
                gold_to_pred[j] = pred_idx
                return True
        return False

    return sum(augment(i, set()) for i in range(len(preds)))


def call_f1(preds: list, golds: list, catalog: dict) -> tuple[float, int]:
    if not golds:
        return (1.0 if not preds else 0.0), 0
    matched = max_matches(preds, golds, catalog)
    return (2 * matched / (len(preds) + len(golds)) if matched else 0.0), matched


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train-jsonl", type=Path, required=True)
    parser.add_argument("--catalog", type=Path, required=True)
    parser.add_argument("--submission", type=Path, required=True)
    args = parser.parse_args()
    catalog = load_catalog(args.catalog)
    predictions = {}
    for line in args.submission.read_text(encoding="utf-8").splitlines():
        if line.strip():
            item = json.loads(line)
            predictions[item["id"]] = item.get("tool_calls", [])
    count = 0
    total = 0.0
    exact = 0
    covered = 0
    for line in args.train_jsonl.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        item = json.loads(line)
        if item["id"] not in predictions:
            continue
        g = item.get("label", {}).get("tool_calls", [])
        p = predictions[item["id"]]
        score, matched = call_f1(p, g, catalog)
        count += 1
        total += score
        exact += int(score == 1.0)
        covered += matched
    if not count:
        raise SystemExit("No overlapping TRAIN sample ids.")
    print(f"Samples: {count} | Tool-call F1 (no hash gate): {100 * total / count:.2f}%")
    print(f"Exact tool-call samples: {exact}/{count} | Matched gold calls: {covered}")
    print("WARNING: This is NOT the official score. Cannot infer state-hash match from train labels.")


if __name__ == "__main__":
    main()


### `lifemate/sft.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/lifemate/sft.py
"""Train-only gold replay, sample-level splitting, and assistant-only SFT loss."""
from __future__ import annotations

import hashlib
import json
import random
import time

from .agent import (SYSTEM_PROMPT, _train_transcript, audio_path, build_step_prompt,
                    history_text, short_json)
from .catalog import allowed_tools


def split_records(records, validation_fraction=0.1, seed=42):
    if not 0 < validation_fraction < 1 or len(records) < 2:
        raise ValueError("Need >=2 unique samples and 0 < validation_fraction < 1")
    ids = [r["id"] for r in records]
    if len(set(ids)) != len(ids):
        raise ValueError("Duplicate sample IDs")
    ordered = sorted(records, key=lambda r: str(r["id"]))
    random.Random(seed).shuffle(ordered)
    count = min(len(ordered) - 1, max(1, round(len(ordered) * validation_fraction)))
    return ordered[count:], ordered[:count]


def replay_sample(item, catalog, env, split_dir, asr=None, max_tool_details=35):
    """Read ONLY train labels; observe real ToolEnv returns before later targets.

    Gold lists must be in execution order. No invented tool results, test labels,
    or direct access to environment files. Fail on ambiguous/invisible targets.
    """
    calls = item.get("label", {}).get("tool_calls")
    if not isinstance(calls, list):
        raise ValueError(f"{item['id']}: missing label.tool_calls list (empty list is valid)")
    started = time.monotonic()
    transcript = _train_transcript(item)
    if transcript is None:
        if asr is None:
            raise ValueError(f"{item['id']}: audio_text missing; pass --asr-model to transcribe train audio")
        transcript = asr.transcribe(audio_path(split_dir, item["audio"]))
    class MissingASR:
        def transcribe(self, path):
            raise ValueError("History audio needs --asr-model or a train history transcript")
    history = history_text(item.get("history", []), split_dir, asr or MissingASR(), started, float("inf"))
    context = item.get("context", {})
    enabled = allowed_tools(catalog, context.get("installed_apps", {}))
    observations, examples = [], []
    last_tool = None
    recent_reads, epoch = set(), 0

    def example(target, step):
        tools, prompt = build_step_prompt(transcript, context, history, enabled, observations,
                                          last_tool, max_tool_details)
        if target["action"] == "call" and target["name"] not in {t["name"] for t in tools["tool_schemas"]}:
            raise ValueError(f"{item['id']} step {step}: target not exposed; increase --max-tool-details consistently in training/inference")
        return {"sample_id": item["id"], "step": step, "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": json.dumps(target, ensure_ascii=False, separators=(",", ":"))},
        ]}

    for step, call in enumerate(calls):
        name, args = call.get("name"), call.get("arguments")
        if name not in enabled or not isinstance(args, dict):
            raise ValueError(f"{item['id']} step {step}: unavailable tool or non-object arguments")
        if name == "sys.ask_missing_info":
            if step != len(calls) - 1 or args.get("tool_name") not in enabled or args.get("tool_name") == name:
                raise ValueError(f"{item['id']}: invalid or non-terminal ask_missing_info")
        validation = env.validate(name, args)
        if not isinstance(validation, dict) or not validation.get("ok"):
            raise ValueError(f"{item['id']} step {step}: gold schema invalid: {validation}")
        read_key = (epoch, name, short_json(args))
        if enabled[name].get("kind") == "read" and read_key in recent_reads:
            raise ValueError(f"{item['id']}: duplicate read rejected by inference policy")
        examples.append(example({"action": "call", "name": name, "arguments": args}, step))
        try:
            result = env.call(name, args)
        except Exception as exc:
            # Match the inference agent's observation exactly, including failed tool calls.
            result = {"ok": False, "error": repr(exc)}
        observations.append({"name": name, "arguments": args, "result": short_json(result, 4500)})
        last_tool = name
        if enabled[name].get("kind") == "read":
            recent_reads.add(read_key)
        elif enabled[name].get("kind") == "write":
            epoch += 1
            recent_reads.clear()
    if not calls or calls[-1]["name"] != "sys.ask_missing_info":
        examples.append(example({"action": "done"}, len(calls)))
    return examples


def encode_example(example, tokenizer, max_length):
    messages = example["messages"]
    if [m["role"] for m in messages] != ["system", "user", "assistant"]:
        raise ValueError("SFT expects system/user/assistant messages")
    prefix = tokenizer.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)
    if full[:len(prefix)] != prefix or len(full) <= len(prefix):
        raise ValueError("Chat template cannot isolate assistant tokens safely")
    if len(full) > max_length:
        raise ValueError(f"{example['sample_id']} step {example['step']}: {len(full)} tokens > max_length={max_length}; increase length, never truncate gold JSON")
    return {"input_ids": full, "attention_mask": [1] * len(full),
            "labels": [-100] * len(prefix) + full[len(prefix):]}


class SupervisedDataset:
    def __init__(self, examples, tokenizer, max_length):
        self.rows = [encode_example(e, tokenizer, max_length) for e in examples]
        if not self.rows:
            raise ValueError("Empty SFT dataset")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


class AssistantCollator:
    def __init__(self, pad_token_id):
        self.pad = pad_token_id

    def __call__(self, rows):
        import torch
        length = max(len(r["input_ids"]) for r in rows)
        return {key: torch.tensor([r[key] + [padding] * (length - len(r[key])) for r in rows])
                for key, padding in (("input_ids", self.pad), ("attention_mask", 0), ("labels", -100))}


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


### `prepare_sft.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/prepare_sft.py
#!/usr/bin/env python3
"""Convert TRAIN gold calls to observed next-action dialogues; split by sample ID."""
import argparse
import json
from pathlib import Path
import sys

from main import resolve_data_layout
from lifemate.catalog import load_catalog
from lifemate.sft import file_sha256, replay_sample, split_records


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--data-root", type=Path, required=True)
    p.add_argument("--output-dir", type=Path, required=True)
    p.add_argument("--validation-fraction", type=float, default=0.1)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--max-tool-details", type=int, default=35)
    p.add_argument("--asr-model", type=Path)
    p.add_argument("--device", choices=["cuda", "cpu"], default="cuda")
    p.add_argument("--asr-compute-type", default="float16")
    args = p.parse_args()
    split_dir, env_root, holder = resolve_data_layout(args.data_root, "train")
    source = split_dir / "train.jsonl"
    records = [json.loads(line) for line in source.read_text(encoding="utf-8").splitlines() if line.strip()]
    train, validation = split_records(records, args.validation_fraction, args.seed)
    if args.output_dir.exists() and any(args.output_dir.iterdir()):
        raise FileExistsError("Use an empty output directory; prepared data is never overwritten")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    sys.path.insert(0, str(split_dir))
    from zaic_tools import ToolEnv
    catalog_path = split_dir / "train_tools_catalog.json"
    catalog = load_catalog(catalog_path)
    asr = None
    if args.asr_model:
        from lifemate.models import LocalASR
        asr = LocalASR(args.asr_model, device=args.device, compute_type=args.asr_compute_type)
    metadata = dict(source=str(source), source_sha256=file_sha256(source),
                    catalog_sha256=file_sha256(catalog_path), seed=args.seed,
                    max_tool_details=args.max_tool_details, partitions={})
    for partition, rows in (("train", train), ("validation", validation)):
        path = args.output_dir / f"{partition}.jsonl"
        count = 0
        # Partial preparations stay marked .partial and are never consumed by the trainer.
        with path.with_suffix(".partial").open("w", encoding="utf-8") as f:
            for i, record in enumerate(rows):
                env = ToolEnv("train", record["id"], root=str(env_root))
                for example in replay_sample(record, catalog, env, split_dir, asr, args.max_tool_details):
                    f.write(json.dumps(example, ensure_ascii=False) + "\n")
                    count += 1
                print(f"{partition}: {i+1}/{len(rows)} samples, {count} SFT steps", flush=True)
        path.with_suffix(".partial").replace(path)
        metadata["partitions"][partition] = dict(sample_ids=[r["id"] for r in rows],
            samples=len(rows), examples=count, sha256=file_sha256(path))
    (args.output_dir / "manifest.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Prepared TRAIN-only SFT data:", args.output_dir)
    if holder is not None:
        holder.cleanup()


if __name__ == "__main__":
    main()


### `train_lora.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/train_lora.py
#!/usr/bin/env python3
"""Assistant-only Unsloth LoRA/QLoRA; optional Transformers CPU test backend."""
import argparse
import json
import logging
import os
from pathlib import Path

from lifemate.sft import AssistantCollator, SupervisedDataset, file_sha256


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--base-model", type=Path, required=True, help="Local model directory (stage HF weights first)")
    p.add_argument("--sft-dir", type=Path, required=True)
    p.add_argument("--output-dir", type=Path, required=True)
    p.add_argument("--backend", choices=["unsloth", "transformers"], default="unsloth")
    p.add_argument("--quantization", choices=["4bit", "none"], default="4bit")
    p.add_argument("--max-length", type=int, default=8192)
    p.add_argument("--epochs", type=float, default=3)
    p.add_argument("--max-steps", type=int, default=-1)
    p.add_argument("--batch-size", type=int, default=1)
    p.add_argument("--gradient-accumulation", type=int, default=8)
    p.add_argument("--learning-rate", type=float, default=2e-4)
    p.add_argument("--lora-r", type=int, default=16)
    p.add_argument("--lora-alpha", type=int, default=32)
    p.add_argument("--lora-dropout", type=float, default=0.0, help="0 enables Unsloth optimized LoRA kernels")
    p.add_argument("--save-steps", type=int, default=100)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--resume-from-checkpoint", type=Path)
    args = p.parse_args()
    # Unsloth must patch Transformers/PEFT BEFORE either is imported.
    FastLanguageModel = None
    if args.backend == "unsloth":
        try:
            from unsloth import FastLanguageModel
        except ImportError as exc:
            raise RuntimeError("Install requirements-train.txt in a supported CUDA environment for Unsloth") from exc
    import torch
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                              Trainer, TrainingArguments, set_seed)
    if not args.base_model.is_dir():
        raise FileNotFoundError(args.base_model)
    if args.output_dir.exists() and any(args.output_dir.iterdir()) and not args.resume_from_checkpoint:
        raise FileExistsError("Use an empty training output directory or --resume-from-checkpoint")
    if args.resume_from_checkpoint and not (args.resume_from_checkpoint / "trainer_state.json").is_file():
        raise FileNotFoundError("Resume requires a Trainer checkpoint including optimizer/scheduler state")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, handlers=[logging.StreamHandler(),
        logging.FileHandler(args.output_dir / "training.log", encoding="utf-8")])
    set_seed(args.seed)
    cuda = torch.cuda.is_available()
    if args.backend == "unsloth" and not cuda:
        raise RuntimeError("Unsloth backend requires CUDA; CPU tests must explicitly use --backend transformers --quantization none")
    if args.quantization == "4bit" and not cuda:
        raise RuntimeError("QLoRA requires CUDA; use --quantization none for a CPU smoke test")
    bf16 = cuda and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16 else (torch.float16 if cuda else torch.float32)
    if args.backend == "unsloth":
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=str(args.base_model.resolve()), max_seq_length=args.max_length,
            dtype=dtype, load_in_4bit=args.quantization == "4bit",
            local_files_only=True, trust_remote_code=False,
            use_exact_model_name=True, fast_inference=False,
        )
    else:
        tokenizer = AutoTokenizer.from_pretrained(str(args.base_model), local_files_only=True, trust_remote_code=False)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        raise ValueError("Tokenizer needs an existing pad/eos token")
    manifest = json.loads((args.sft_dir / "manifest.json").read_text())
    data = {}
    ids = {}
    for part in ("train", "validation"):
        path = args.sft_dir / f"{part}.jsonl"
        if file_sha256(path) != manifest["partitions"][part]["sha256"]:
            raise ValueError(f"Prepared {part} data changed after preparation")
        examples = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
        ids[part] = {e["sample_id"] for e in examples}
        if ids[part] != set(manifest["partitions"][part]["sample_ids"]):
            raise ValueError(f"{part} sample IDs disagree with manifest")
        data[part] = SupervisedDataset(examples, tokenizer, args.max_length)
    if ids["train"] & ids["validation"]:
        raise ValueError("Train/validation sample leakage")
    run_config = {
        **{k: str(v.resolve()) if isinstance(v, Path) else v for k, v in vars(args).items()},
        "torch": torch.__version__, "train_steps": len(data["train"]),
        "validation_steps": len(data["validation"]), "data_manifest": manifest,
    }
    if args.resume_from_checkpoint:
        original_path = args.resume_from_checkpoint.parent / "training_config.json"
        if not original_path.is_file():
            raise FileNotFoundError("Resume needs the original training_config.json next to checkpoint directories")
        original = json.loads(original_path.read_text(encoding="utf-8"))
        for key in ("backend", "base_model", "data_manifest", "quantization", "max_length",
                    "batch_size", "gradient_accumulation", "learning_rate", "lora_r",
                    "lora_alpha", "lora_dropout", "seed"):
            if original.get(key) != run_config[key]:
                raise ValueError(f"Resume configuration mismatch: {key}")
    # Save before training so interrupted runs can still be resumed safely.
    config_path = args.output_dir / "training_config.json"
    config_path.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")
    targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    if args.backend == "unsloth":
        model = FastLanguageModel.get_peft_model(model, r=args.lora_r, lora_alpha=args.lora_alpha,
            lora_dropout=args.lora_dropout, bias="none", target_modules=targets,
            use_gradient_checkpointing="unsloth", random_state=args.seed,
            use_rslora=False, loftq_config=None)
    else:
        kwargs = dict(local_files_only=True, trust_remote_code=False, torch_dtype=dtype,
                      attn_implementation="sdpa")
        if args.quantization == "4bit":
            kwargs.update(quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=dtype), device_map={"": int(os.environ.get("LOCAL_RANK", 0))})
        model = AutoModelForCausalLM.from_pretrained(str(args.base_model), **kwargs)
        if args.quantization == "4bit":
            model = prepare_model_for_kbit_training(model, gradient_checkpointing_kwargs={"use_reentrant": False})
        config = LoraConfig(r=args.lora_r, lora_alpha=args.lora_alpha,
            lora_dropout=args.lora_dropout, bias="none", task_type="CAUSAL_LM", target_modules=targets)
        model = get_peft_model(model, config)
    model.config.use_cache = False
    model.print_trainable_parameters()
    training_args = TrainingArguments(output_dir=str(args.output_dir),
        num_train_epochs=args.epochs, max_steps=args.max_steps,
        per_device_train_batch_size=args.batch_size, per_device_eval_batch_size=1,
        gradient_accumulation_steps=args.gradient_accumulation,
        learning_rate=args.learning_rate, lr_scheduler_type="cosine", warmup_ratio=0.03,
        weight_decay=0.01, max_grad_norm=1.0, optim="adamw_torch",
        bf16=bf16, fp16=cuda and not bf16, use_cpu=not cuda,
        gradient_checkpointing=args.backend != "unsloth", gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=1, eval_strategy="steps", eval_steps=args.save_steps,
        save_strategy="steps", save_steps=args.save_steps, save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        prediction_loss_only=True, report_to="none", seed=args.seed, data_seed=args.seed,
        dataloader_num_workers=0, remove_unused_columns=False)
    trainer = Trainer(model=model, args=training_args, processing_class=tokenizer,
        train_dataset=data["train"], eval_dataset=data["validation"],
        data_collator=AssistantCollator(tokenizer.pad_token_id))
    train_result = trainer.train(resume_from_checkpoint=str(args.resume_from_checkpoint) if args.resume_from_checkpoint else None)
    trainer.save_metrics("train", train_result.metrics)
    metrics = trainer.evaluate()
    trainer.save_metrics("eval", metrics)
    trainer.save_state()
    adapter = args.output_dir / "adapter"
    trainer.save_model(str(adapter))
    tokenizer.save_pretrained(str(adapter))
    run_config["best_checkpoint"] = trainer.state.best_model_checkpoint
    config_path.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved adapter and tokenizer: {adapter}; validation metrics: {metrics}")


if __name__ == "__main__":
    main()


### `merge_lora.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/merge_lora.py
#!/usr/bin/env python3
"""Export a local LoRA adapter as a standalone offline model (no bitsandbytes)."""
import argparse
from pathlib import Path


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--base-model", type=Path, required=True)
    p.add_argument("--adapter", type=Path, required=True)
    p.add_argument("--output-dir", type=Path, required=True)
    p.add_argument("--dtype", choices=["float32", "float16", "bfloat16"], default="float16")
    args = p.parse_args()
    if args.output_dir.exists() and any(args.output_dir.iterdir()):
        raise FileExistsError("Merged output directory must be empty")
    if args.output_dir.resolve() in (args.base_model.resolve(), args.adapter.resolve()):
        raise ValueError("Output cannot overwrite base model or adapter")
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    model = AutoModelForCausalLM.from_pretrained(str(args.base_model),
        torch_dtype=getattr(torch, args.dtype), device_map="cpu", local_files_only=True,
        trust_remote_code=False, low_cpu_mem_usage=True)
    model = PeftModel.from_pretrained(model, str(args.adapter), local_files_only=True)
    model = model.merge_and_unload(safe_merge=True)
    model.save_pretrained(str(args.output_dir), safe_serialization=True, max_shard_size="4GB")
    tokenizer_path = args.adapter if (args.adapter / "tokenizer_config.json").is_file() else args.base_model
    tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path), local_files_only=True, trust_remote_code=False)
    tokenizer.save_pretrained(str(args.output_dir))
    print("Merged offline model:", args.output_dir)


if __name__ == "__main__":
    main()


### `finetune.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/finetune.py
#!/usr/bin/env python3
"""One command: resolve HF weights -> replay TRAIN -> LoRA/QLoRA -> optional merge."""
import argparse
import json
import os
from pathlib import Path
import subprocess
import sys


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--data-root", type=Path, required=True)
    p.add_argument("--base-model", required=True, help="Local directory or Hugging Face model ID")
    p.add_argument("--revision", default="main", help="HF commit or revision; resolved SHA is logged")
    p.add_argument("--output-dir", type=Path, required=True)
    p.add_argument("--asr-model", type=Path, help="Optional fallback for train audio/history without text")
    p.add_argument("--asr-device", choices=["cuda", "cpu"], default="cuda")
    p.add_argument("--asr-compute-type", default="float16")
    p.add_argument("--validation-fraction", type=float, default=0.1)
    p.add_argument("--max-tool-details", type=int, default=35)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--export-merged", action="store_true", help="Needs RAM/disk for full-precision base + adapter")
    # Remaining args are forwarded to train_lora.py, e.g. --max-steps or --quantization.
    args, training_options = p.parse_known_args()
    if args.output_dir.exists() and any(args.output_dir.iterdir()):
        raise FileExistsError("Choose an empty run directory. To resume training use train_lora.py directly.")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    app = Path(__file__).resolve().parent
    env = os.environ.copy()
    env.update(PYTHONUNBUFFERED="1", PYTHONDONTWRITEBYTECODE="1", TOKENIZERS_PARALLELISM="false")
    def run(script, options):
        command = [sys.executable, str(app / script), *map(str, options)]
        with (args.output_dir / (Path(script).stem + ".log")).open("w", encoding="utf-8") as log:
            with subprocess.Popen(command, env=env, cwd=app, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True, errors="replace") as child:
                for line in child.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                if child.wait():
                    raise RuntimeError(f"{script} failed; see {log.name}")
    base = Path(args.base_model).expanduser()
    source = {"local": str(base.resolve())}
    if not base.is_dir():
        from huggingface_hub import HfApi, snapshot_download
        sha = HfApi().model_info(args.base_model, revision=args.revision).sha
        base = Path(snapshot_download(args.base_model, revision=sha))
        source = {"repo": args.base_model, "revision": sha, "local": str(base)}
    (args.output_dir / "base_model.json").write_text(json.dumps(source, indent=2), encoding="utf-8")
    # Offline after staging. Training/preparation will not contact model registries.
    env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1")
    prepare = ["--data-root", args.data_root, "--output-dir", args.output_dir / "sft",
               "--validation-fraction", args.validation_fraction, "--seed", args.seed,
               "--max-tool-details", args.max_tool_details]
    if args.asr_model:
        prepare.extend(["--asr-model", args.asr_model, "--device", args.asr_device,
                        "--asr-compute-type", args.asr_compute_type])
    run("prepare_sft.py", prepare)
    run("train_lora.py", [*training_options, "--base-model", base, "--sft-dir", args.output_dir / "sft",
                          "--output-dir", args.output_dir / "training", "--seed", args.seed])
    if args.export_merged:
        run("merge_lora.py", ["--base-model", base, "--adapter", args.output_dir / "training/adapter",
                              "--output-dir", args.output_dir / "merged"])
    print("Fine-tune completed:", args.output_dir)


if __name__ == "__main__":
    main()


### `tests/test_baseline.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/tests/test_baseline.py
import json
from pathlib import Path

from lifemate.agent import AgentConfig, LifeMateAgent, parse_decision
from lifemate.catalog import allowed_tools
from main import resolve_data_layout
from score_train import call_f1

CATALOG = {
    "smart_home.get_status": {
        "name": "smart_home.get_status", "kind": "read", "description": "Liệt kê thiết bị nhà",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
    "smart_home.light_set": {
        "name": "smart_home.light_set", "kind": "write", "description": "Bật/tắt đèn",
        "parameters": {"type": "object", "properties": {
            "device_id": {"type": "string", "x-ref": "smart_home.devices"},
            "power": {"type": "string", "enum": ["on", "off"]},
        }, "required": ["device_id"]},
    },
    "sys.ask_missing_info": {
        "name": "sys.ask_missing_info", "kind": "meta", "description": "Hỏi người dùng",
        "parameters": {"type": "object", "properties": {"tool_name": {"type": "string"}, "params": {"type": "array"}},
                       "required": ["tool_name", "params"]},
    },
}


class StubASR:
    def transcribe(self, path):
        return "Bật đèn phòng khách"


class StubLLM:
    def __init__(self, outputs):
        self.outputs = iter(outputs)
    def generate(self, system, user):
        return next(self.outputs)


class FakeEnv:
    def __init__(self):
        self.calls = []
    def validate(self, name, args):
        spec = CATALOG[name]["parameters"]
        missing = set(spec.get("required", [])) - set(args)
        return {"ok": not missing, "error": "missing required" if missing else ""}
    def call(self, name, args):
        self.calls.append({"name": name, "arguments": args})
        if name == "smart_home.get_status":
            return {"ok": True, "devices": [{"id": "light_living", "name": "đèn phòng khách"}]}
        return {"ok": True}
    def get_tool_calls(self):
        return self.calls


def item():
    return {"id": "train_0001", "audio": "audio/train_0001/train_0001.wav",
            "context": {"timezone": "Asia/Ho_Chi_Minh", "installed_apps": {
                "smart_home": {"tools": ["smart_home.get_status", "smart_home.light_set"]}}}, "history": []}


def run(outputs, installed=None):
    entry = item()
    if installed is not None:
        entry["context"]["installed_apps"] = installed
    env = FakeEnv()
    result = LifeMateAgent(StubASR(), StubLLM(outputs), CATALOG, AgentConfig()).run(
        entry, env, Path("/tmp/lifemate_test_no_access_to_env"), "train")
    return env.calls, result


def test_multiple_steps_read_write():
    calls, stats = run([
        '{"action":"call","name":"smart_home.get_status","arguments":{}}',
        '{"action":"call","name":"smart_home.light_set","arguments":{"device_id":"light_living","power":"on"}}',
        '{"action":"done"}',
    ])
    assert [c["name"] for c in calls] == ["smart_home.get_status", "smart_home.light_set"]
    assert stats["executed"] == 2


def test_schema_invalid_proposal_not_executed():
    calls, stats = run([
        '{"action":"call","name":"smart_home.light_set","arguments":{"power":"on"}}',
        '{"action":"call","name":"smart_home.get_status","arguments":{}}',
        '{"action":"done"}',
    ])
    assert len(calls) == 1
    assert stats["invalid_proposals"] == 1


def test_unsupported_app_rejected():
    calls, _ = run(['{"action":"call","name":"smart_home.get_status","arguments":{}}',
                    '{"action":"done"}'], installed={})
    assert calls == []


def test_missing_info():
    calls, _ = run([
        '{"action":"call","name":"sys.ask_missing_info","arguments":{"tool_name":"smart_home.light_set","params":["device_id"]}}'
    ])
    assert calls[0]["name"] == "sys.ask_missing_info"


def test_json_parser_and_out_of_scope():
    assert parse_decision('```json\n{"action":"done"}\n```') == {"action": "done"}
    assert allowed_tools(CATALOG, {}).keys() == {"sys.ask_missing_info"}


def test_proxy_matching():
    gold = [{"name": "smart_home.get_status", "arguments": {}},
            {"name": "smart_home.light_set", "arguments": {"device_id": "light_living", "power": "on"}}]
    assert call_f1(list(reversed(gold)), gold, CATALOG)[0] == 1.0
    assert call_f1([], [], CATALOG)[0] == 1.0
    assert call_f1(gold[:1], gold, CATALOG)[0] == 2 / 3


def test_direct_split_layout(tmp_path):
    (tmp_path / "public_test.jsonl").write_text("", encoding="utf-8")
    split_dir, root, temp = resolve_data_layout(tmp_path, "public_test")
    assert split_dir == tmp_path
    assert (root / "public_test").resolve() == tmp_path.resolve()
    temp.cleanup()


def test_parent_split_layout(tmp_path):
    (tmp_path / "public_test").mkdir()
    (tmp_path / "public_test" / "public_test.jsonl").write_text("", encoding="utf-8")
    split_dir, root, temp = resolve_data_layout(tmp_path, "public_test")
    assert split_dir == tmp_path / "public_test"
    assert root == tmp_path
    assert temp is None


### `tests/test_sft.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/tests/test_sft.py
import json
from pathlib import Path

import pytest

from lifemate.sft import encode_example, replay_sample, split_records, AssistantCollator
from test_baseline import CATALOG, FakeEnv, item


def labeled():
    entry = item()
    entry['audio_text'] = 'Bật đèn phòng khách'
    entry['label'] = {'tool_calls': [
        {'name': 'smart_home.get_status', 'arguments': {}},
        {'name': 'smart_home.light_set', 'arguments': {'device_id': 'light_living', 'power': 'on'}},
    ]}
    return entry


def test_gold_replay_observations_do_not_leak_future_targets():
    env = FakeEnv()
    steps = replay_sample(labeled(), CATALOG, env, Path('/unused'))
    assert len(steps) == 3
    payloads = [json.loads(s['messages'][1]['content']) for s in steps]
    assert payloads[0]['previous_actions_and_results'] == []
    assert 'light_living' not in steps[0]['messages'][1]['content']
    assert 'light_living' in payloads[1]['previous_actions_and_results'][0]['result']
    assert json.loads(steps[-1]['messages'][-1]['content']) == {'action': 'done'}
    assert len(env.calls) == 2


def test_schema_and_missing_labels_fail_instead_of_becoming_done():
    entry = labeled()
    entry.pop('label')
    with pytest.raises(ValueError, match='missing label'):
        replay_sample(entry, CATALOG, FakeEnv(), Path('/unused'))
    entry = labeled()
    entry['label']['tool_calls'][1]['arguments'] = {}
    with pytest.raises(ValueError, match='gold schema invalid'):
        replay_sample(entry, CATALOG, FakeEnv(), Path('/unused'))


def test_terminal_missing_info_does_not_add_done():
    entry = labeled()
    entry['label']['tool_calls'] = [{'name': 'sys.ask_missing_info',
        'arguments': {'tool_name': 'smart_home.light_set', 'params': ['device_id']}}]
    assert len(replay_sample(entry, CATALOG, FakeEnv(), Path('/unused'))) == 1


def test_split_is_grouped_deterministic_and_order_independent():
    records = [{'id': str(i)} for i in range(20)]
    train, valid = split_records(records)
    assert not {r['id'] for r in train} & {r['id'] for r in valid}
    assert (train, valid) == split_records(records[::-1])
    with pytest.raises(ValueError, match='Duplicate'):
        split_records([{'id': 'x'}, {'id': 'x'}])


class TemplateTokenizer:
    def apply_chat_template(self, messages, tokenize, add_generation_prompt):
        return [1, 2, 3] if add_generation_prompt else [1, 2, 3, 7, 8, 9]


def test_loss_masks_prompt_and_padding_and_rejects_truncated_json():
    example = {'sample_id': 'train_1', 'step': 0, 'messages': [
        {'role': role, 'content': ''} for role in ('system', 'user', 'assistant')]}
    row = encode_example(example, TemplateTokenizer(), 6)
    assert row['labels'] == [-100, -100, -100, 7, 8, 9]
    with pytest.raises(ValueError, match='never truncate'):
        encode_example(example, TemplateTokenizer(), 5)
    short = {k: v[:-1] for k, v in row.items()}
    batch = AssistantCollator(0)([row, short])
    assert batch['labels'][1, -1].item() == -100
    assert batch['attention_mask'][1, -1].item() == 0


### `tests/test_finetune_integration.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/tests/test_finetune_integration.py
"""Actual tiny random Qwen + LoRA training; no downloads or competition data."""
import json
import os
from pathlib import Path
import subprocess
import sys
import types

import pytest


def test_lora_accepts_modern_torchao_api(monkeypatch):
    """Modern TorchAO no longer exports LinearActivationQuantizedTensor."""
    torch = pytest.importorskip('torch')
    pytest.importorskip('peft')
    from peft.tuners.lora import torchao as dispatch_module
    from peft import LoraConfig, get_peft_model
    from transformers import Qwen3Config, Qwen3ForCausalLM
    fake_utils = types.ModuleType('torchao.utils')
    fake_utils.TorchAOBaseTensor = type('TorchAOBaseTensor', (), {})
    monkeypatch.setitem(sys.modules, 'torchao.utils', fake_utils)
    monkeypatch.setattr(dispatch_module, 'is_torchao_available', lambda: True)
    model = Qwen3ForCausalLM(Qwen3Config(vocab_size=8, hidden_size=16, intermediate_size=32,
        num_hidden_layers=1, num_attention_heads=2, num_key_value_heads=1, head_dim=8))
    model = get_peft_model(model, LoraConfig(r=2, task_type='CAUSAL_LM', target_modules=['q_proj']))
    assert any('lora_B' in name for name, _ in model.named_parameters())


def test_train_adapter_reload_and_merge(tmp_path):
    torch = pytest.importorskip('torch')
    pytest.importorskip('peft')
    from transformers import Qwen3Config, Qwen3ForCausalLM, PreTrainedTokenizerFast, AutoModelForCausalLM
    from tokenizers import Tokenizer, models, pre_tokenizers
    from lifemate.models import LocalLLM
    base = tmp_path / 'tiny-base'
    base.mkdir()
    tok = Tokenizer(models.WordLevel({'[UNK]': 0, '[PAD]': 1, '[EOS]': 2,
        '<s>': 3, '</s>': 4, 'system': 5, 'user': 6, 'assistant': 7}, unk_token='[UNK]'))
    tok.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
    tokenizer = PreTrainedTokenizerFast(tokenizer_object=tok, unk_token='[UNK]',
        pad_token='[PAD]', eos_token='[EOS]', additional_special_tokens=['<s>', '</s>'])
    tokenizer.chat_template = "{% for m in messages %}{{ '<s>' + m['role'] + '\n' + m['content'] + '</s>\n' }}{% endfor %}{% if add_generation_prompt %}{{ '<s>assistant\n' }}{% endif %}"
    tokenizer.model_input_names = ['input_ids', 'attention_mask']
    tokenizer.save_pretrained(base)
    torch.manual_seed(0)
    model = Qwen3ForCausalLM(Qwen3Config(vocab_size=len(tokenizer), hidden_size=16,
        intermediate_size=32, num_hidden_layers=1, num_attention_heads=2, num_key_value_heads=1,
        head_dim=8, max_position_embeddings=4096, eos_token_id=2, pad_token_id=1))
    model.save_pretrained(base)
    data = tmp_path / 'data' / 'train'
    data.mkdir(parents=True)
    (data / 'train_tools_catalog.json').write_text(json.dumps({'apps': [{'tools': [
        {'name': 'home.read', 'kind': 'read', 'parameters': {'type': 'object', 'properties': {}}}]}]}))
    (data / 'train.jsonl').write_text(''.join(json.dumps({
        'id': f'train_{i}', 'audio_text': 'Kiểm tra thiết bị',
        'context': {'installed_apps': {'home': {'tools': ['home.read']}}}, 'history': [],
        'label': {'tool_calls': [{'name': 'home.read', 'arguments': {}}]},
    }) + '\n' for i in range(4)))
    (data / 'zaic_tools.py').write_text('''class ToolEnv:
    def __init__(self, split, id, root): assert split == 'train'
    def validate(self, name, args): return {'ok': True}
    def call(self, name, args): return {'ok': True, 'devices': ['living_room']}
''')
    app = Path(__file__).resolve().parents[1]
    run = tmp_path / 'run'
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '', 'OMP_NUM_THREADS': '1',
           'PYTHONDONTWRITEBYTECODE': '1', 'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'}
    result = subprocess.run([sys.executable, app / 'finetune.py', '--data-root', data.parent,
        '--base-model', base, '--output-dir', run, '--backend', 'transformers', '--quantization', 'none',
        '--max-steps', '2', '--gradient-accumulation', '1', '--save-steps', '1',
        '--max-length', '4096', '--learning-rate', '0.01', '--export-merged'],
        env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=180)
    assert result.returncode == 0, result.stdout
    adapter = run / 'training' / 'adapter'
    assert (adapter / 'adapter_model.safetensors').is_file()
    from safetensors.torch import load_file
    weights = load_file(str(adapter / 'adapter_model.safetensors'))
    assert any(torch.count_nonzero(v).item() > 0 for k, v in weights.items() if 'lora_B' in k)
    metrics = json.loads((run / 'training/eval_results.json').read_text())
    assert 0 < metrics['eval_loss'] < 100
    llm = LocalLLM(base, device='cpu', dtype='float32', adapter_path=adapter, max_new_tokens=2)
    merged = AutoModelForCausalLM.from_pretrained(run / 'merged', local_files_only=True, torch_dtype=torch.float32)
    llm.model.eval()
    merged.eval()
    with torch.no_grad():
        tokens = torch.tensor([[3, 6, 0, 4]])
        a = llm.model(tokens).logits
        b = merged(tokens).logits
    assert torch.allclose(a, b, atol=0.003, rtol=0.01)
    assert isinstance(llm.generate('system', 'user'), str)
    assert (run / 'train_lora.log').is_file()
    assert (run / 'training/checkpoint-2/trainer_state.json').is_file()
    resume = subprocess.run([sys.executable, app / 'train_lora.py', '--base-model', base,
        '--sft-dir', run / 'sft', '--output-dir', run / 'training', '--backend', 'transformers', '--quantization', 'none',
        '--max-steps', '3', '--gradient-accumulation', '1', '--save-steps', '1',
        '--max-length', '4096', '--learning-rate', '0.01',
        '--resume-from-checkpoint', run / 'training/checkpoint-2'],
        env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=180)
    assert resume.returncode == 0, resume.stdout
    assert (run / 'training/checkpoint-3/trainer_state.json').is_file()


### `tests/test_audio.py`


In [ ]:
%%writefile /kaggle/working/lifemate_app/tests/test_audio.py
"""Exercise the real faster-whisper/PyAV audio decoder without model weights."""
import wave

import numpy as np
import pytest


def test_whisper_decodes_and_resamples_stereo_wav(tmp_path):
    decode_audio = pytest.importorskip('faster_whisper.audio').decode_audio
    path = tmp_path / 'speech.wav'
    rate = 8000
    samples = (2000 * np.sin(2 * np.pi * 440 * np.arange(rate // 2) / rate)).astype('<i2')
    with wave.open(str(path), 'wb') as f:
        f.setnchannels(2)
        f.setsampwidth(2)
        f.setframerate(rate)
        f.writeframes(np.column_stack([samples, samples]).tobytes())
    # Calls av.open(metadata_errors='ignore') inside faster-whisper, then resamples.
    audio = decode_audio(str(path), sampling_rate=16000)
    assert audio.shape == (8000,)
    assert audio.dtype == np.float32
    assert np.isfinite(audio).all()
    assert 0.01 < np.max(np.abs(audio)) < 1


In [ ]:
run_logged([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"],
           OUT / "unit_tests.log", cwd=APP, env=BASE_ENV)
toolenv_check_script = """
import sys
from pathlib import Path
from main import resolve_data_layout
split_dir, env_root, holder = resolve_data_layout(Path(sys.argv[1]), sys.argv[2])
try:
    sys.path.insert(0, str(split_dir))
    from zaic_tools import ToolEnv
    env = ToolEnv(sys.argv[2], sys.argv[3], root=str(env_root))
    required = ('validate', 'call', 'get_tool_calls', 'get_state_hash')
    missing = [name for name in required if not callable(getattr(env, name, None))]
    if missing:
        raise RuntimeError('ToolEnv instance is missing callable API methods: ' + ', '.join(missing))
    print('ToolEnv instance public API OK')
finally:
    if holder is not None:
        holder.cleanup()
"""
run_logged([sys.executable, "-c", toolenv_check_script, SPLIT_DIR, SPLIT, EXPECTED_IDS[0]],
           OUT / "setup.log", cwd=APP, env=BASE_ENV)


## 4. Tải Hugging Face model và chuyển ASR nếu cần

Hai model mặc định là public, không yêu cầu token. Với model cần xác thực, thêm Kaggle Secret rồi đặt `HF_SECRET_NAME`; token chỉ truyền qua environment, không ghi vào cấu hình/logs. SHA thực tế được lưu để tái lập lần chạy. `LOCAL_ASR` luôn phải trỏ tới model CTranslate2 đã chuyển đổi. Không tải model trong bước inference.


In [ ]:
download_settings = dict(asr_repo=ASR_REPO, llm_repo=LLM_REPO, asr_format=ASR_FORMAT,
    asr_revision=ASR_REVISION, llm_revision=LLM_REVISION,
    local_asr=LOCAL_ASR, local_llm=LOCAL_LLM, model_root=str(MODEL_ROOT), out=str(OUT))
(OUT / "download_settings.json").write_text(json.dumps(download_settings, indent=2), encoding="utf-8")
DOWNLOAD_ENV = BASE_ENV.copy()
DOWNLOAD_ENV.update(HF_HUB_OFFLINE="0", TRANSFORMERS_OFFLINE="0", HF_DATASETS_OFFLINE="0")
DOWNLOAD_ENV["HF_HUB_DISABLE_XET"] = "1"  # dùng HTTP download chuẩn của Hub
if HF_SECRET_NAME:
    from kaggle_secrets import UserSecretsClient
    DOWNLOAD_ENV["HF_TOKEN"] = UserSecretsClient().get_secret(HF_SECRET_NAME)

download_script = r'''
import json, sys, os
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
cfg = json.loads(Path(sys.argv[1]).read_text())
root = Path(cfg["model_root"])
api = HfApi()
manifest = {}
for part in ("asr", "llm"):
    local = cfg["local_" + part]
    if local:
        target = Path(local).expanduser().resolve()
        if not target.is_dir():
            raise FileNotFoundError(target)
        manifest[part] = {"path": str(target), "source": "local"}
        continue
    repo = cfg[part + "_repo"]
    requested = cfg[part + "_revision"] or "main"
    sha = api.model_info(repo, revision=requested).sha
    target = root / part / sha
    print(f"Downloading {part}: {repo}@{sha}", flush=True)
    snapshot_download(repo_id=repo, revision=sha, local_dir=str(target), max_workers=4)
    if part == "asr" and cfg["asr_format"] == "transformers":
        from ctranslate2.converters import TransformersConverter
        converted = root / "asr_ct2" / sha
        if not (converted / ".conversion_complete").is_file():
            # Rebuild only notebook-generated incomplete conversion, never user LOCAL_ASR.
            import shutil
            if converted.exists():
                shutil.rmtree(converted)
            converted.parent.mkdir(parents=True, exist_ok=True)
            converter = TransformersConverter(str(target))
            converter.convert(str(converted), quantization="float16")
            from transformers import AutoTokenizer
            AutoTokenizer.from_pretrained(str(target), use_fast=True, local_files_only=True).save_pretrained(str(converted))
            if (target / "preprocessor_config.json").is_file():
                shutil.copy2(target / "preprocessor_config.json", converted / "preprocessor_config.json")
            if not (converted / "tokenizer.json").is_file():
                raise FileNotFoundError("ASR conversion needs a local fast tokenizer.json")
            (converted / ".conversion_complete").touch()
        target = converted
    manifest[part] = {"path": str(target), "repo": repo, "revision": sha, "source": "huggingface"}
asr = Path(manifest["asr"]["path"])
llm = Path(manifest["llm"]["path"])
for file in (asr / "model.bin", asr / "config.json", llm / "config.json"):
    if not file.is_file():
        raise FileNotFoundError(f"Missing model file: {file}")
if not any(llm.glob("*.safetensors")):
    raise FileNotFoundError("LLM cần trọng số safetensors trong thư mục model.")
# Verify tokenizer loads locally now, before the long inference step.
from transformers import AutoTokenizer
AutoTokenizer.from_pretrained(str(llm), local_files_only=True, trust_remote_code=False)
Path(cfg["out"], "model_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Both model directories are staged.")
'''
download_path = APP / "download_kaggle_models.py"
download_path.write_text(download_script, encoding="utf-8")
try:
    run_logged([sys.executable, download_path, OUT / "download_settings.json"],
               OUT / "models.log", env=DOWNLOAD_ENV, cwd=APP)
finally:
    # Không giữ token trong dictionary dùng cho suy luận hoặc file artifacts.
    DOWNLOAD_ENV.pop("HF_TOKEN", None)
MODEL_MANIFEST = json.loads((OUT / "model_manifest.json").read_text())
ASR_PATH = Path(MODEL_MANIFEST["asr"]["path"])
LLM_PATH = Path(MODEL_MANIFEST["llm"]["path"])
print("ASR:", ASR_PATH, "\nLLM:", LLM_PATH)


## 5. Fine-tune tool calling trên TRAIN (LoRA/QLoRA)

Mặc định `TRAIN_LLM=True`: huấn luyện LoRA/QLoRA, rồi tự chuyển sang **base model + adapter vừa train** cho task chính. Nhãn phải là `label.tool_calls` theo thứ tự thực thi. Replay dùng `ToolEnv` thật của train; kết quả tool trước được đưa vào prompt bước sau. Chỉ tính loss trên assistant JSON, split theo sample ID trước khi tạo các bước, không đọc nhãn test.

Muốn dùng adapter đã có: đặt `TRAIN_LLM=False`, `LOCAL_ADAPTER` tới thư mục adapter. Muốn dùng model đã merge: đặt `TRAIN_LLM=False`, `LOCAL_ADAPTER=None`, `LOCAL_LLM` tới model merged. Nếu thiếu train data khi bật training, cell sẽ dừng rõ ràng thay vì dùng model gốc âm thầm.

Backend huấn luyện mặc định là **Unsloth FastLanguageModel** với QLoRA 4-bit, LoRA dropout=0 và gradient checkpointing `unsloth`. Import Unsloth trước Transformers/PEFT trong subprocess riêng. T4 dùng FP16; GPU hỗ trợ thì BF16. Sequence dài có thể hết VRAM: điều chỉnh cấu hình cho GPU thực tế. JSON target quá dài sẽ bị báo lỗi, không bị cắt âm thầm. `eval_loss` là loss trên validation theo teacher forcing, **không phải điểm F1 chính thức**. Checkpoint/adapter nằm dưới `/kaggle/working/lifemate_finetunes`; không nhét model nhiều GB vào ZIP logs. Nếu mất session, gắn lại base weights + adapter hoặc dùng checkpoint để resume bằng `train_lora.py`.


In [ ]:
ADAPTER_PATH = Path(LOCAL_ADAPTER) if LOCAL_ADAPTER else None
if TRAIN_LLM:
    if TRAIN_DATA_ROOT:
        training_data = Path(TRAIN_DATA_ROOT)
    else:
        training_candidates = discover_split("/kaggle/input", "train")
        if len(training_candidates) != 1:
            raise ValueError(f"Cần đúng một tập train: {training_candidates}. Đặt TRAIN_DATA_ROOT rõ ràng.")
        training_data = training_candidates[0]
    FT_RUN = Path("/kaggle/working/lifemate_finetunes") / datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    FT_ENV = BASE_ENV.copy()
    FT_ENV.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1")
    train_command = [sys.executable, "-u", APP / "finetune.py",
        "--data-root", training_data, "--base-model", LLM_PATH, "--output-dir", FT_RUN,
        "--backend", TRAIN_BACKEND,
        "--validation-fraction", str(TRAIN_VALIDATION_FRACTION),
        "--max-tool-details", str(MAX_TOOL_DETAILS), "--seed", str(SEED),
        "--epochs", str(TRAIN_EPOCHS), "--max-steps", str(TRAIN_MAX_STEPS),
        "--max-length", str(TRAIN_MAX_LENGTH),
        "--gradient-accumulation", str(TRAIN_GRAD_ACCUMULATION),
        "--quantization", "4bit" if HW["gpu"] else "none"]
    if TRAIN_USE_ASR:
        train_command.extend(["--asr-model", ASR_PATH, "--asr-device", DEVICE,
                              "--asr-compute-type", ASR_COMPUTE])
    try:
        run_logged(train_command, OUT / "training_pipeline.log", cwd=APP, env=FT_ENV)
    finally:
        if FT_RUN.is_dir():
            for filename in ("prepare_sft.log", "train_lora.log", "base_model.json",
                             "training/training_config.json", "training/eval_results.json",
                             "training/train_results.json", "training/trainer_state.json", "sft/manifest.json"):
                path = FT_RUN / filename
                if path.is_file():
                    shutil.copy2(path, OUT / ("training_" + filename.replace("/", "_")))
    ADAPTER_PATH = FT_RUN / "training" / "adapter"
if ADAPTER_PATH is not None and not (ADAPTER_PATH / "adapter_config.json").is_file():
    raise FileNotFoundError(f"Không có adapter hợp lệ: {ADAPTER_PATH}")
print("Inference LLM:", LLM_PATH, "adapter:", ADAPTER_PATH)


## 6. Chạy toàn bộ split → kiểm chứng → lưu kết quả và logs

Inference sử dụng local-only model, giữ soft budget và thuật toán của baseline. stdout/stderr được stream ra cell và ghi vào `inference.log`; `diagnostics.jsonl` có transcript, thời gian, số calls và lỗi của từng mẫu. Notebook kiểm tra đủ ID, thứ tự và schema output; không tự suy diễn tính đúng của state hash. Nếu lỗi, file kết quả một phần và logs vẫn được đóng gói để chẩn đoán. Chạy lại cell tạo thư mục run mới, giữ kết quả cũ.


In [ ]:
# Chạy lại cell này không ghi đè kết quả cũ.
if (OUT / "inference.log").exists():
    previous_out = OUT
    OUT = OUT.parent / datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    OUT.mkdir(parents=True)
    for name in ("setup.log", "models.log", "hardware.log", "unit_tests.log", "environment.txt", "model_manifest.json"):
        if (previous_out / name).is_file():
            shutil.copy2(previous_out / name, OUT / name)
    for path in previous_out.glob("training_*"):
        if path.is_file():
            shutil.copy2(path, OUT / path.name)
INFER_ENV = BASE_ENV.copy()
INFER_ENV.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1")
INFER_ENV.pop("HF_TOKEN", None)
SUBMISSION = OUT / "submission.jsonl"
DIAGNOSTICS = OUT / "diagnostics.jsonl"
command = [sys.executable, "-u", APP / "main.py",
    "--data-root", SPLIT_DIR, "--split", SPLIT,
    "--asr-model", ASR_PATH, "--llm-model", LLM_PATH,
    "--output", SUBMISSION, "--log-path", DIAGNOSTICS,
    "--device", DEVICE, "--asr-compute-type", ASR_COMPUTE, "--llm-dtype", LLM_DTYPE,
    "--limit", str(LIMIT), "--sample-seconds", str(SAMPLE_SECONDS),
    "--max-input-tokens", str(MAX_INPUT_TOKENS), "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--max-calls", str(MAX_CALLS), "--max-invalid", str(MAX_INVALID),
    "--max-tool-details", str(MAX_TOOL_DETAILS), "--asr-beam", str(ASR_BEAM), "--seed", str(SEED)]
if ADAPTER_PATH is not None:
    command.extend(["--llm-adapter", ADAPTER_PATH])
if FAIL_FAST:
    command.append("--fail-fast")
if USE_TRAIN_TRANSCRIPT:
    command.append("--use-train-transcript")
run_config = dict(split=SPLIT, data_root=str(SPLIT_DIR), sample_count=len(EXPECTED_IDS),
                  limit=LIMIT, hardware=HW, command=[str(x) for x in command], models=MODEL_MANIFEST)
(OUT / "run_config.json").write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")
validation = {"status": "running", "expected_rows": len(EXPECTED_IDS)}
try:
    run_logged(command, OUT / "inference.log", cwd=APP, env=INFER_ENV)
    def read_jsonl(path):
        with path.open(encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]
    rows = read_jsonl(SUBMISSION)
    diagnostics = read_jsonl(DIAGNOSTICS)
    if [r.get("id") for r in rows] != EXPECTED_IDS:
        raise ValueError("Submission thiếu, thừa hoặc sai thứ tự ID.")
    if [r.get("id") for r in diagnostics] != EXPECTED_IDS:
        raise ValueError("Diagnostics không đủ ID.")
    for row in rows:
        if set(row) != {"id", "tool_calls", "state_hash"}:
            raise ValueError(f"Sai schema output: {row.get('id')}")
        if not isinstance(row["tool_calls"], list) or not isinstance(row["state_hash"], str) or not row["state_hash"]:
            raise ValueError(f"Invalid calls/hash: {row['id']}")
    errors = [r["id"] for r in diagnostics if "error" in r]
    times = [r["sample_seconds"] for r in diagnostics]
    validation.update(status="completed" if not errors else "completed_with_sample_errors",
        rows=len(rows), error_count=len(errors), error_ids=errors,
        mean_seconds=sum(times) / len(times), max_seconds=max(times),
        samples_over_30s=sum(t > 30 for t in times),
        submission_sha256=hashlib.sha256(SUBMISSION.read_bytes()).hexdigest())
    if errors:
        raise RuntimeError(f"Đã xuất đủ mẫu nhưng {len(errors)} mẫu lỗi; xem diagnostics và inference.log.")
    if SPLIT == "train" and RUN_TRAIN_SCORE:
        run_logged([sys.executable, APP / "score_train.py", "--train-jsonl", SPLIT_DIR / "train.jsonl",
                    "--catalog", CATALOG_PATH, "--submission", SUBMISSION],
                   OUT / "train_score.log", cwd=APP, env=INFER_ENV)
    print(json.dumps(validation, ensure_ascii=False, indent=2))
except BaseException as exc:
    if validation["status"] != "completed_with_sample_errors":
        validation["status"] = "failed"
    validation["failure"] = str(exc)
    raise
finally:
    (OUT / "validation.json").write_text(json.dumps(validation, ensure_ascii=False, indent=2), encoding="utf-8")
    archive = OUT / "lifemate_artifacts.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(OUT.iterdir()):
            if path.is_file() and path != archive:
                bundle.write(path, arcname=path.name)
    print("\nKết quả và logs:", OUT)
    print("ZIP:", archive)


In [ ]:
from IPython.display import display, FileLink
for name in ("submission.jsonl", "diagnostics.jsonl", "inference.log", "validation.json", "lifemate_artifacts.zip"):
    path = OUT / name
    if path.is_file():
        display(FileLink(str(path)))
print("Có thể tải toàn bộ thư mục run từ tab Output của Kaggle sau khi Save Version / Run All.")
